# Re-extract Bad Frames (ShuttleSet GDINO Skeletons)

Re-extract **3,007 specific frames** whose skeletons fall outside the court boundary
`x=[450, 1450], y=[200, 1000]` on 1920×1080 frames.

Uses a **tighter court boundary** than the initial extraction to fix residual
contamination. The bad frames are hardcoded from a prior scan — no re-scanning needed.

For each bad frame, reloads the image, re-runs GDINO+YOLO with the tighter spatial
filter, and overwrites the corresponding `(t_idx, :)` slice in `skeletons.npy`.

In [ ]:
import os, sys, zipfile

# ── Colab setup ───────────────────────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_PATH     = '/content/drive/MyDrive/FineBadminton/baddiev2_colab.zip'
    PROJECT_PATH = '/content/Baddiev2'

    if not os.path.exists(PROJECT_PATH):
        print("Extracting project files...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print("Done.")
    else:
        print("Project already extracted.")

    os.chdir(os.path.join(PROJECT_PATH, 'notebooks'))
    sys.path.insert(0, PROJECT_PATH)
    print(f"CWD: {os.getcwd()}")
else:
    print("Local run")

In [ ]:
!pip install -q tqdm transformers timm pillow ultralytics

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import cv2
import torch
from pathlib import Path
from tqdm import tqdm

from src.config import PROJECT_ROOT, SS_FRAMES, SS_SKELETONS_GDINO

## 1. Court Boundary & Thresholds

Tighter boundary than initial extraction to eliminate residual contamination.

In [ ]:
# ── Court boundary (pixels, 1920×1080) ────────────────────────────────────────
X_MIN, X_MAX = 450, 1450
Y_MIN, Y_MAX = 200, 1000

_X_MIN_FRAC = X_MIN / 1920  # 0.234
_X_MAX_FRAC = X_MAX / 1920  # 0.755
_Y_MIN_FRAC = Y_MIN / 1080  # 0.185
_Y_MAX_FRAC = Y_MAX / 1080  # 0.926
_MIN_H_FRAC = 0.10

_BOX_THRESH   = 0.30
_TXT_THRESH   = 0.25
_IOU_THRESH   = 0.25
_GDINO_PROMPT = 'badminton player .'

print(f"Court boundary: x=[{X_MIN},{X_MAX}], y=[{Y_MIN},{Y_MAX}]")
print(f"Fractions: x=[{_X_MIN_FRAC:.3f},{_X_MAX_FRAC:.3f}], y=[{_Y_MIN_FRAC:.3f},{_Y_MAX_FRAC:.3f}]")

## 2. Bad Frames (hardcoded from scan)

Each entry is `(t_index_in_array, frame_num)` per match.

In [ ]:
BAD_FRAMES = {
    "Anders_ANTONSEN_Jonatan_CHRISTIE Indonesia_Masters_2020_QuarterFinals": [(7, 13850), (52, 17056), (71, 19699), (97, 21887), (105, 22540), (109, 22635), (122, 24099), (134, 24326), (154, 25633), (160, 26801), (170, 27600), (178, 27805), (180, 27847), (195, 31297), (203, 32656), (221, 33034), (227, 33178), (264, 37218), (276, 37516), (282, 37641), (289, 39130), (316, 41910), (317, 41921), (352, 44816), (370, 47518), (386, 48823), (442, 56382), (502, 61993), (535, 64787), (549, 66209), (645, 82396), (665, 83409), (666, 83420), (685, 85139), (704, 87533), (721, 89781), (729, 90558), (771, 92687), (781, 92951), (797, 94821), (823, 96165), (829, 96938), (848, 99065), (878, 102310), (890, 102602), (919, 105281), (928, 106188), (942, 108484), (944, 108533), (945, 108542), (954, 108784), (1005, 114591), (1006, 114620), (1014, 115228)],
    "Anders_Antonsen_Viktor_Axelsen_HSBC_BWF_WORLD_TOUR_FINALS_2020_Finals": [(11, 16819), (34, 16899), (35, 16903), (36, 16907), (37, 16911), (38, 16912), (39, 16915), (53, 16967), (54, 16971), (63, 17003), (64, 17007), (102, 17662), (103, 17666), (125, 17746), (126, 17750), (130, 17766), (132, 17774), (140, 17802), (141, 17806), (142, 17810), (171, 18496), (181, 18536), (190, 18564), (191, 18568), (195, 18580), (196, 18584), (218, 19105), (222, 19121), (223, 19125), (226, 19133), (227, 19137), (228, 19141), (260, 19807), (262, 19811), (263, 19815), (269, 19835), (288, 19903), (296, 19931), (297, 19935), (298, 19939), (299, 19941), (300, 19943), (346, 20107), (349, 20119), (355, 20139), (358, 20151), (361, 20163), (364, 20171), (366, 20179), (367, 20183), (395, 20283), (397, 20291), (399, 20299), (400, 20303), (401, 20307), (403, 20311), (407, 21022), (408, 21026), (414, 21046), (462, 21837), (463, 21841), (468, 21861), (470, 21867), (471, 21869), (480, 21902), (482, 21909), (483, 21913), (508, 22001), (509, 22005), (510, 22009), (548, 22149), (549, 22153), (550, 22157), (551, 22161), (553, 22169), (554, 22774), (555, 22778), (556, 22782), (562, 22802), (574, 23427), (575, 23431), (576, 23433), (577, 23435), (578, 23439), (579, 23443), (587, 23471), (588, 23475), (590, 23479), (591, 23483), (601, 23519), (602, 23523), (632, 24927), (634, 24935), (635, 24939), (636, 24940), (637, 24943), (650, 24991), (651, 24995), (652, 24999), (653, 25003), (656, 25668), (657, 25672), (658, 25676), (659, 25680), (660, 25683), (661, 25684), (662, 25688), (663, 25692), (664, 25696), (665, 25700), (666, 25704), (667, 25708), (668, 25712), (669, 25716), (670, 25720), (671, 25724), (672, 25728), (673, 25732), (674, 25736), (675, 25740), (681, 25764), (682, 25768), (683, 25772), (684, 25776), (685, 25780), (686, 25784), (704, 25852), (706, 25860), (708, 25868), (709, 25872), (724, 25928), (726, 25933), (727, 25936), (732, 25956), (735, 25964), (736, 25968), (741, 25988), (742, 25992), (746, 26004), (747, 26008), (748, 26010), (749, 26012), (755, 26035), (756, 26036), (757, 26040), (759, 26048), (763, 26064), (773, 26100), (788, 26148), (820, 26264), (824, 26276), (848, 27430), (849, 27434), (850, 27438), (852, 27442), (854, 27450), (858, 27465), (859, 27466), (867, 27498), (872, 27514), (873, 27518), (891, 28716), (892, 28720), (899, 28744), (900, 28748), (901, 28750), (917, 28808), (919, 28816), (923, 28828), (953, 29431), (958, 29451), (963, 29467), (971, 29495), (997, 29587), (1012, 29639), (1022, 29675), (1025, 29683), (1038, 29735), (1044, 30368), (1048, 30380), (1049, 30384), (1050, 30388), (1094, 30548), (1097, 30560), (1111, 30612), (1125, 30660), (1126, 30664), (1128, 30670), (1146, 30736), (1147, 30738), (1148, 30740), (1149, 30744), (1152, 30756), (1160, 30784), (1161, 30786), (1177, 30844), (1178, 30848), (1214, 30976), (1215, 30980), (1246, 32590), (1247, 32594), (1276, 32710), (1277, 32714), (1278, 32718), (1279, 32722), (1308, 33357), (1312, 33373), (1313, 33377), (1315, 33381), (1323, 33409), (1324, 33413), (1336, 33453), (1337, 33457), (1338, 33461), (1339, 33465), (1340, 33469), (1342, 33477), (1343, 33481), (1346, 33493), (1367, 34601), (1377, 36566), (1383, 36587), (1386, 36598), (1387, 36602), (1394, 36626), (1407, 36674), (1408, 36675), (1409, 36678), (1410, 36682), (1445, 38249), (1446, 38253), (1447, 38255), (1448, 38257), (1460, 38301), (1461, 38305), (1462, 38309), (1466, 38321), (1477, 38365), (1479, 38373), (1480, 38377), (1491, 38421), (1504, 39559), (1505, 39563), (1509, 39577), (1510, 39579), (1514, 39595), (1515, 39599), (1521, 39619), (1522, 39622), (1523, 39623), (1524, 39627), (1525, 39631), (1526, 39635), (1527, 39639), (1532, 39657), (1534, 39663), (1556, 39739), (1557, 39743), (1565, 39771), (1576, 39807), (1577, 39811), (1602, 40587), (1604, 40595), (1605, 40599), (1606, 40601), (1607, 40603), (1619, 41216), (1620, 41220), (1621, 41224), (1622, 41228), (1623, 41232), (1624, 41236), (1643, 41749), (1644, 41753), (1645, 41757), (1646, 41761), (1647, 41764), (1648, 41765), (1649, 41769), (1650, 41773), (1651, 41777), (1652, 41781), (1653, 41785), (1654, 41789), (1678, 41877), (1680, 41883), (1681, 41885), (1707, 41977), (1708, 41981), (1715, 42009), (1765, 43128), (1783, 43192), (1784, 43196), (1801, 43745), (1803, 43753), (1808, 43769), (1810, 43774), (1811, 43777), (1812, 43781), (1821, 43817), (1861, 44671), (1862, 44675), (1863, 44679), (1864, 44680), (1866, 44687), (1867, 44691), (1874, 44715), (1885, 44755), (1902, 46029), (1905, 46041), (1906, 46042), (1909, 46053), (1910, 46055), (1911, 46057), (1917, 46081), (1918, 46082), (1919, 46085), (1920, 46089), (1935, 46759), (1937, 46767), (1939, 46775), (1959, 46851), (1960, 46854), (1961, 46855), (1966, 46875), (1969, 46883), (1977, 46911), (1995, 46975), (1996, 46979), (2000, 46995), (2001, 46999), (2002, 47002), (2055, 48584), (2056, 48587), (2059, 48596), (2062, 48606), (2063, 48608), (2066, 48620), (2068, 48628), (2075, 48652), (2076, 48656), (2105, 48756), (2106, 48760), (2107, 48764), (2114, 48788), (2117, 48800), (2119, 48808), (2122, 48816), (2138, 48876), (2139, 48878), (2142, 48888), (2143, 48892), (2155, 48932), (2182, 49028), (2184, 49036), (2185, 49040), (2186, 49043), (2187, 49044), (2188, 49048), (2189, 49052), (2214, 49148), (2226, 49192), (2234, 49220), (2268, 49994), (2272, 50006), (2275, 50018), (2276, 50022), (2277, 50026), (2278, 50030), (2279, 50034), (2287, 50065), (2289, 50070), (2290, 50074), (2291, 50078), (2293, 51001), (2294, 51005), (2295, 51009), (2296, 51012), (2297, 51013), (2298, 51017), (2299, 51021), (2300, 51025), (2311, 51065), (2322, 51105), (2326, 51117), (2337, 51161), (2342, 51177), (2346, 51193), (2356, 51229), (2357, 51233), (2358, 51236), (2359, 51237), (2361, 51245), (2389, 51337), (2431, 51481), (2433, 51485), (2434, 51489), (2435, 51493), (2436, 51497), (2445, 51529), (2446, 51533), (2481, 51661), (2482, 51665), (2483, 51666), (2484, 51669), (2536, 52687), (2537, 52691), (2540, 53432), (2555, 53488), (2566, 53528), (2567, 53532), (2587, 53608), (2622, 55012), (2623, 55014), (2645, 55092), (2663, 55156), (2666, 55164), (2667, 55168), (2668, 55172), (2677, 55204), (2678, 55208), (2679, 55211), (2680, 55212), (2681, 55216), (2695, 55268), (2696, 55272), (2697, 55275), (2698, 55276), (2699, 55280), (2709, 55316), (2711, 55324), (2757, 55492), (2779, 59661), (2780, 59665), (2781, 59669), (2782, 59673), (2783, 59676), (2784, 59677), (2785, 59681), (2787, 59689), (2788, 59693), (2789, 59695), (2790, 59697), (2791, 59701), (2792, 59705), (2793, 59709), (2796, 59721), (2797, 59725), (2800, 59737), (2801, 59739), (2802, 59741), (2804, 59749), (2805, 59753), (2809, 59767), (2810, 59769), (2811, 59773), (2813, 59781), (2814, 59785), (2815, 59788), (2816, 59789), (2818, 59797), (2819, 59801), (2820, 59805), (2821, 59809), (2822, 59813), (2823, 59817), (2824, 59821), (2825, 59824), (2826, 59825), (2827, 59829), (2828, 59833), (2829, 59837), (2830, 59841), (2831, 59845), (2832, 59849), (2833, 59853), (2836, 59861), (2837, 59865), (2839, 59873), (2840, 59877), (2841, 59881), (2842, 59885), (2843, 59889), (2855, 59929), (2856, 59933), (2857, 59937), (2860, 59945), (2864, 59961), (2865, 59965), (2873, 59993), (2876, 60005), (2877, 60009), (2878, 60013), (2879, 60017), (2880, 60021), (2881, 60025), (2882, 60028), (2883, 60029), (2884, 60033), (2885, 60037), (2886, 60041), (2895, 60069), (2896, 60073), (2897, 60077), (2899, 60085), (2901, 60092), (2902, 60093), (2903, 60097), (2905, 60105), (2907, 60113), (2908, 60117), (2909, 60121), (2914, 60141), (2916, 60149), (2917, 60153), (2920, 60165), (2921, 60169), (2922, 60173), (2923, 60177), (2924, 60180), (2925, 60181), (2926, 60185), (2927, 60189), (2928, 60193), (2929, 60197), (2930, 60201), (2931, 60205), (2932, 60209), (2933, 60213), (2934, 60217), (2935, 60221), (2936, 60225), (2937, 60229), (2938, 60231), (2949, 60269), (2959, 60982), (2965, 61002), (2973, 61546), (2974, 61550), (2975, 61554), (2976, 61558), (2977, 61562), (2978, 61563), (2981, 61574), (2982, 61578), (2990, 61603), (2997, 61627), (2998, 61630), (3000, 61638), (3011, 62117), (3013, 62124), (3022, 62153), (3025, 62165), (3034, 62572), (3035, 62576), (3037, 62584), (3038, 62588), (3041, 62596), (3042, 62600), (3043, 62604), (3044, 62608), (3045, 62612), (3050, 62632), (3054, 62648), (3055, 62650), (3056, 62652), (3057, 62656), (3058, 62660), (3072, 63124), (3079, 63148), (3084, 63164), (3087, 63176), (3088, 63180), (3089, 63184), (3090, 63188), (3091, 63192), (3096, 63212), (3097, 63216), (3099, 63224), (3102, 63236), (3103, 63240), (3107, 63252), (3108, 63256), (3110, 63742), (3124, 63790), (3127, 63802), (3128, 63803), (3134, 63822), (3135, 63826), (3136, 63830), (3137, 63834), (3138, 63838), (3139, 63842), (3140, 63846), (3141, 63849), (3142, 63850), (3144, 63858), (3145, 63862), (3146, 63866), (3151, 63886), (3152, 63890), (3155, 63898), (3157, 63906), (3161, 63922), (3167, 63942), (3171, 63958), (3172, 63962), (3173, 63966), (3174, 63970), (3175, 63974), (3176, 63978), (3177, 63982), (3178, 63985), (3179, 63986), (3180, 63990), (3181, 63994), (3182, 63998), (3183, 64444), (3184, 64448), (3197, 64492), (3198, 64496), (3204, 64520), (3205, 64524), (3219, 64576), (3220, 64580), (3221, 64584), (3224, 64995), (3225, 64999), (3226, 65002), (3227, 65003), (3228, 65007), (3231, 65019), (3233, 65027), (3234, 65031), (3235, 65035), (3236, 65335), (3237, 65339), (3240, 65350), (3242, 65355), (3244, 65363), (3245, 65367), (3246, 65371), (3247, 65375), (3248, 65379), (3250, 65383), (3252, 65391), (3253, 65395), (3254, 66502), (3256, 66510), (3257, 66514), (3258, 66517), (3259, 66518), (3260, 66522), (3261, 66526), (3262, 66530), (3263, 66534), (3264, 66538), (3265, 66542), (3268, 66552), (3269, 66554), (3270, 66558), (3271, 66562), (3272, 66566), (3273, 67388), (3285, 67428), (3286, 67432), (3289, 69251), (3290, 69255), (3292, 69263), (3293, 69266), (3294, 69267), (3296, 69275), (3302, 69295), (3311, 70138), (3312, 70142), (3318, 70164), (3319, 70166), (3321, 70174), (3322, 70178), (3324, 70186), (3328, 70202), (3329, 70206), (3330, 70210), (3331, 70213), (3332, 70214), (3333, 70218), (3334, 70222), (3335, 70226), (3336, 70230), (3338, 70238), (3347, 70270), (3348, 70274), (3349, 70278), (3350, 70282), (3351, 70286), (3352, 70290), (3353, 70294), (3360, 70318), (3362, 70326), (3363, 70330), (3365, 70338), (3366, 70342), (3367, 70346), (3369, 70354), (3370, 70358), (3371, 70362), (3372, 70744), (3373, 70748), (3374, 70752), (3375, 70756), (3378, 70764), (3382, 71124), (3383, 71128), (3384, 71132), (3385, 71135), (3387, 71140), (3388, 71144), (3389, 71148), (3390, 71152), (3392, 71160), (3393, 71164), (3394, 71168), (3395, 71172), (3396, 71176), (3397, 71180), (3398, 71184), (3399, 71188), (3400, 71192), (3401, 71196), (3402, 71200), (3404, 71208), (3405, 71212), (3406, 71213), (3410, 71228), (3411, 71586), (3412, 71590), (3430, 71654), (3432, 71662), (3434, 71670), (3437, 71682), (3438, 71686), (3439, 71690), (3441, 71698), (3443, 71706), (3444, 71710), (3445, 71714), (3446, 71718), (3447, 71722), (3448, 71726), (3449, 71730), (3452, 71742), (3454, 71746), (3458, 72166), (3459, 72170), (3460, 72174), (3461, 72178), (3462, 72181), (3463, 72182), (3464, 72186), (3465, 72190), (3466, 72194), (3467, 72195), (3470, 72206), (3471, 72210), (3472, 72214), (3473, 72218), (3474, 72221), (3475, 72222), (3476, 72226), (3480, 72242), (3481, 72246), (3485, 72262), (3487, 72270), (3488, 72274), (3489, 72278), (3490, 72282), (3497, 72306), (3498, 72310), (3505, 72721), (3509, 72737), (3510, 72741), (3514, 72753), (3515, 72757), (3516, 72761), (3517, 72765), (3520, 72777), (3521, 72781), (3528, 72805), (3529, 72809), (3530, 72813), (3535, 72829), (3536, 72833), (3538, 72841), (3547, 72873), (3548, 72877), (3549, 72881), (3550, 72885), (3551, 72889), (3552, 72893), (3553, 72896), (3555, 72901), (3557, 72909), (3563, 73491), (3564, 73495), (3565, 73499), (3566, 73503), (3567, 73507), (3568, 73511), (3569, 73515), (3570, 73519), (3571, 73523), (3572, 73527), (3573, 73531), (3574, 73535), (3575, 73536), (3576, 73539), (3577, 73543), (3583, 73567), (3584, 73571), (3585, 73575), (3587, 73583), (3588, 73587), (3592, 73599), (3596, 73615), (3597, 73619), (3599, 73627), (3600, 73631), (3601, 73635), (3602, 73639), (3603, 73643), (3604, 73647), (3605, 73650), (3606, 73651), (3607, 73655), (3608, 73659), (3609, 73663), (3610, 73667), (3613, 73679), (3617, 73695), (3619, 74140), (3620, 74144), (3621, 74148), (3622, 74152), (3623, 74155), (3625, 74160), (3626, 74164), (3627, 74168), (3636, 74200), (3641, 74220), (3644, 74228), (3645, 74232), (3647, 74240), (3648, 74244), (3650, 74252), (3659, 74284), (3660, 74288), (3661, 74292), (3662, 74296), (3663, 74299), (3666, 74308), (3670, 75340), (3671, 75344), (3673, 75348), (3674, 75352), (3675, 75356), (3680, 75372), (3681, 75376), (3682, 75380), (3683, 75384), (3684, 75388), (3685, 75392), (3686, 75394), (3687, 75396), (3688, 75400), (3689, 75404), (3690, 75408), (3691, 75412), (3692, 75414), (3693, 75416), (3694, 75420), (3695, 75424), (3696, 75428), (3697, 75430), (3698, 75432), (3699, 75436), (3700, 75440), (3701, 75444), (3702, 75841), (3703, 75845), (3704, 75849), (3705, 75853), (3713, 75881), (3714, 75882), (3717, 75893), (3718, 75897), (3720, 76291), (3731, 76327), (3740, 76359), (3744, 76751), (3745, 76755), (3746, 76759), (3747, 76763), (3748, 76766), (3752, 76779), (3756, 77203), (3758, 77207), (3761, 77219), (3768, 77243), (3769, 77247), (3770, 77251), (3805, 80944), (3806, 80948), (3820, 81000), (3840, 81072), (3841, 81076), (3843, 81084), (3854, 81120), (3893, 81800), (3894, 81804), (3927, 81920), (3928, 81924), (3952, 82008), (3954, 82016), (3961, 82040), (3975, 82756), (3978, 82768), (3981, 82780), (4003, 83348), (4018, 83404), (4021, 83412), (4022, 83416), (4027, 83432), (4028, 83436), (4071, 83588), (4074, 83600), (4075, 83604), (4076, 83608), (4077, 83612), (4078, 83616), (4079, 83620), (4096, 84448), (4098, 84456), (4099, 84460), (4100, 84464), (4101, 84468), (4130, 84576), (4131, 84580), (4132, 84584), (4153, 84664), (4166, 84712), (4172, 85412), (4173, 85416), (4174, 85420), (4175, 85424), (4176, 85427), (4177, 85428), (4190, 85476), (4191, 85480), (4192, 85484), (4193, 85486), (4194, 85488), (4195, 85492), (4196, 85496), (4197, 85500), (4198, 85504), (4199, 85508), (4200, 85511), (4201, 85512), (4202, 85516), (4203, 85520), (4204, 85524), (4205, 86244), (4206, 86248), (4213, 86272), (4254, 86412), (4255, 86416), (4256, 86420), (4257, 86424), (4258, 86428), (4272, 86478), (4273, 86480), (4280, 86508), (4293, 87309), (4296, 87317), (4297, 87321), (4309, 87365), (4310, 87369), (4319, 87397), (4348, 87501), (4350, 87509), (4357, 87533), (4363, 87557), (4364, 87561), (4365, 87565), (4396, 87681), (4400, 87697), (4402, 87701), (4403, 87705), (4407, 87721), (4410, 87729), (4414, 87745), (4431, 87805), (4458, 90039), (4483, 90127), (4485, 90135), (4486, 90139), (4487, 90142), (4488, 90143), (4490, 90151), (4491, 90155), (4492, 90157), (4493, 90159), (4494, 90163), (4496, 90171), (4501, 90191), (4506, 90211), (4511, 90227), (4514, 90239), (4515, 90243), (4516, 90247), (4517, 90251), (4518, 90255), (4528, 90991), (4529, 90995), (4535, 91015), (4536, 91019), (4538, 91027), (4539, 91031), (4540, 91035), (4546, 91055), (4559, 91103), (4560, 91107), (4563, 91115), (4564, 91119), (4583, 91935), (4586, 91943), (4588, 91951), (4589, 91955), (4609, 92027), (4613, 92043), (4614, 92047), (4618, 92059), (4648, 92171), (4654, 92191), (4657, 92203), (4659, 92211), (4660, 92213), (4661, 92215), (4665, 93693), (4666, 93697), (4667, 93701), (4668, 93705), (4669, 93708), (4670, 93709), (4671, 93713), (4672, 93717), (4673, 93721), (4674, 93725), (4675, 93729), (4699, 94325), (4725, 94417), (4726, 94421), (4733, 94445), (4735, 94452), (4736, 94453), (4737, 94457), (4748, 94497), (4758, 94535), (4785, 94633), (4791, 94653), (4814, 95398), (4859, 95562), (4861, 95570), (4865, 95586), (4866, 95590), (4867, 95594), (4868, 95596), (4869, 95598), (4870, 95602), (4872, 95610), (4873, 95614), (4874, 95618), (4876, 95626), (4878, 95634), (4879, 95638), (4880, 95641), (4881, 95642), (4882, 95646), (4884, 95654), (4891, 95678), (4954, 96581), (4967, 96629), (4980, 96673), (4981, 96677), (5004, 96757), (5007, 96769), (5008, 96773), (5016, 96805), (5026, 96841), (5055, 96945), (5059, 96961), (5075, 97017), (5078, 97029), (5106, 97129), (5117, 97169), (5120, 97180), (5131, 98861), (5132, 98865), (5133, 98869), (5134, 98873), (5135, 98876), (5136, 98877), (5137, 98881), (5138, 98885), (5139, 98889), (5140, 98893), (5141, 98897), (5148, 98921), (5150, 98929), (5151, 98933), (5152, 98937), (5153, 98939), (5154, 98941), (5155, 98945), (5156, 98949), (5183, 100020), (5185, 100028), (5196, 100068), (5224, 100168), (5225, 100172), (5226, 100175), (5233, 102927), (5234, 102931), (5235, 102934), (5241, 102955), (5242, 102958), (5244, 102963), (5245, 102967), (5248, 102979), (5249, 102983), (5250, 102987), (5251, 102991), (5254, 102999), (5255, 103003), (5256, 103007), (5257, 103011), (5280, 103087), (5282, 103095), (5283, 103099), (5284, 103103), (5288, 103117), (5291, 103127), (5292, 103131), (5293, 103616), (5294, 103620), (5295, 103624), (5296, 103628), (5299, 103636), (5300, 103640), (5304, 103652), (5308, 103668), (5309, 103671), (5310, 103672), (5311, 103676), (5312, 103680), (5315, 103692), (5316, 103696), (5317, 103700), (5321, 103712), (5322, 103716), (5323, 103718), (5328, 103736), (5329, 103740), (5330, 103742), (5331, 103744), (5332, 103748), (5333, 103752), (5335, 103760), (5345, 103796), (5346, 103800), (5347, 103804), (5351, 103820), (5352, 103824), (5353, 103828), (5354, 103832), (5356, 103840), (5363, 103864), (5369, 103885), (5376, 103912), (5377, 103916), (5378, 103920), (5379, 103922), (5380, 103924), (5381, 103928), (5382, 103932), (5383, 103936), (5384, 103940), (5392, 103968), (5393, 103972), (5394, 103976), (5395, 105127), (5396, 105131), (5397, 105135), (5398, 105139), (5401, 105147), (5402, 105151), (5403, 105155), (5404, 105159), (5405, 105163), (5406, 105167), (5407, 105171), (5408, 105175), (5409, 105179), (5410, 105183), (5411, 105187), (5414, 105199), (5415, 105203), (5416, 105207), (5417, 105210), (5418, 105211), (5421, 105223), (5422, 105227), (5423, 105231), (5424, 105235), (5425, 105236), (5426, 105239), (5427, 105243), (5429, 105251), (5430, 105255), (5432, 105263), (5433, 105267), (5434, 105271), (5440, 105291), (5442, 105299), (5445, 105311), (5446, 105315), (5449, 105323), (5451, 105331), (5455, 105347), (5456, 105351), (5457, 105355), (5458, 105359), (5463, 106262), (5469, 106282), (5473, 106294), (5478, 106314), (5480, 106322), (5481, 106326), (5484, 107028), (5493, 107060), (5494, 107061), (5495, 107064), (5500, 107084), (5501, 107088), (5502, 107092), (5506, 107104), (5507, 107108), (5509, 107116), (5510, 107120), (5514, 107132), (5515, 107136), (5519, 107662), (5522, 107670), (5529, 107694), (5531, 107702), (5532, 107706), (5533, 107710), (5534, 107714), (5542, 107742), (5543, 107746), (5546, 107758), (5548, 107766), (5549, 107770), (5550, 107774), (5551, 107778), (5552, 107782), (5553, 107786), (5555, 107792), (5556, 107794), (5557, 107798), (5558, 107802), (5561, 107814), (5562, 107818), (5563, 107822), (5566, 107834), (5567, 107838), (5568, 107839), (5569, 107842), (5570, 107846), (5573, 107858), (5576, 107866), (5577, 107870), (5578, 107874), (5582, 107890), (5583, 107894), (5587, 107907), (5588, 107910), (5590, 107918), (5591, 107922), (5600, 109297), (5601, 109301), (5606, 109317), (5607, 109321), (5614, 109810), (5618, 109825), (5621, 109834), (5627, 109854), (5628, 109858), (5629, 109862), (5630, 109866), (5631, 109870), (5632, 109874), (5633, 109878), (5634, 109882), (5637, 109894), (5638, 109898), (5642, 109910), (5643, 109914), (5644, 109918), (5647, 109926), (5648, 109930), (5675, 110723), (5676, 110727), (5681, 110743), (5682, 110747), (5683, 110751), (5687, 110763), (5689, 110771), (5690, 110775), (5691, 110779), (5692, 110780), (5694, 110787), (5695, 110791), (5696, 110795), (5697, 110799), (5698, 110803), (5699, 110805), (5708, 110835), (5711, 110847), (5712, 110851), (5714, 110859), (5715, 110863), (5716, 110865), (5717, 110867), (5718, 110871), (5719, 110875), (5720, 110879), (5721, 111581), (5722, 111585), (5723, 111589), (5724, 111593), (5725, 111596), (5727, 111601), (5728, 111605), (5729, 111609), (5732, 111621), (5733, 111625), (5737, 111641), (5744, 111665), (5745, 111669), (5746, 111673), (5748, 111677), (5749, 111681), (5750, 111685), (5751, 111689), (5771, 111761), (5773, 111769), (5774, 111773), (5776, 111779), (5777, 111781), (5778, 111785), (5779, 111789), (5780, 111791), (5782, 111797), (5783, 111801), (5784, 111805), (5786, 111813), (5787, 111814), (5788, 111817), (5789, 111821), (5790, 111825), (5798, 111855), (5799, 111857), (5800, 111861), (5802, 111867), (5804, 111873), (5805, 111877), (5806, 111881), (5808, 111889), (5815, 111913), (5821, 111933), (5825, 111949), (5827, 111957), (5828, 111961), (5829, 111965), (5833, 111977), (5845, 112021), (5846, 112025), (5853, 112968), (5855, 112976), (5862, 113000), (5865, 113012), (5866, 113013), (5867, 113016), (5868, 113020), (5874, 113044), (5877, 113056), (5885, 113088), (5886, 113089), (5899, 113140), (5902, 113148), (5903, 113152), (5904, 113156), (5905, 113866), (5906, 113870), (5907, 113874), (5908, 113878), (5911, 113886), (5912, 113890), (5914, 113898), (5915, 113902), (5916, 113904), (5917, 113906), (5923, 113930), (5924, 113934), (5925, 113938), (5926, 113941), (5927, 113942), (5928, 113946), (5929, 113950), (5930, 113954), (5932, 113962), (5933, 113966), (5939, 113986), (5940, 113990), (5941, 113994), (5942, 113998), (5943, 114002), (5944, 114006), (5945, 114010), (5946, 114014), (5947, 114018), (5948, 114022), (5949, 114024), (5950, 114026), (5951, 114030), (5952, 114034), (5953, 114038), (5956, 114050), (5958, 114058), (5959, 114061), (5960, 114062), (5961, 114066), (5962, 114070), (5963, 114074), (5964, 114075), (5965, 114078), (5966, 114082), (5967, 114086), (5968, 114090), (5969, 114094), (5970, 114098), (5971, 114099), (5972, 114102), (5973, 114106), (5975, 114114), (5977, 114122), (5980, 114134), (5981, 114138), (5982, 114142), (5983, 114146), (5984, 114150), (5985, 114151), (5986, 114154), (5987, 114158), (5988, 114162), (5989, 114166), (5990, 114170), (5991, 114174), (5992, 114176), (5993, 114178), (5994, 114182), (5995, 114186), (5996, 114190), (5997, 114194), (5998, 114198), (5999, 114201), (6000, 114202), (6001, 114206), (6002, 114210), (6004, 115477), (6005, 115481), (6006, 115485), (6007, 115489), (6008, 115492), (6009, 115493), (6010, 115497), (6011, 115501), (6012, 115505), (6017, 115525), (6018, 115526), (6026, 115553), (6027, 115557), (6028, 115561), (6029, 115565), (6033, 115581), (6034, 115585), (6036, 115591), (6037, 115593), (6038, 115597), (6039, 115601), (6040, 115605), (6053, 116303), (6055, 116311), (6056, 116315), (6058, 116323), (6060, 116331), (6062, 116339), (6063, 116343), (6064, 116346), (6065, 116347), (6066, 116351), (6067, 116355), (6068, 116359), (6069, 116363), (6071, 116371), (6072, 116375), (6073, 116376), (6074, 116379), (6075, 116383), (6077, 116391), (6085, 116419), (6086, 116423), (6087, 116424), (6088, 116427), (6089, 116431), (6090, 116435), (6091, 116439), (6092, 116443), (6093, 116447), (6095, 116455), (6096, 116457), (6097, 116459), (6100, 116471), (6104, 116487), (6105, 116491), (6106, 116495), (6109, 116503), (6120, 116541), (6121, 116543), (6122, 116547), (6123, 116551), (6124, 116555), (6127, 116563), (6134, 116587), (6135, 116591), (6137, 116595), (6141, 116611), (6142, 116615), (6149, 116643), (6157, 116671), (6160, 116683), (6161, 116685), (6162, 116687), (6165, 116699), (6172, 116723), (6183, 116759), (6186, 116771), (6187, 116775), (6188, 116779), (6189, 116783), (6191, 116791), (6192, 116795), (6195, 116807), (6198, 116815), (6200, 116823), (6202, 116831), (6203, 116835), (6204, 116839), (6205, 116843), (6206, 116847), (6217, 116887), (6218, 116891), (6222, 116907), (6224, 116915), (6230, 116935), (6233, 116947), (6234, 116951), (6235, 116955), (6236, 116959), (6237, 116963), (6240, 116975), (6241, 116979), (6242, 116983), (6246, 116999), (6254, 117027), (6267, 117071), (6268, 117075), (6270, 117083), (6271, 117084), (6272, 117087), (6273, 117091), (6282, 117123), (6283, 117127), (6284, 117129), (6285, 117131), (6287, 117139), (6288, 117143), (6290, 117151), (6299, 117187), (6300, 117191), (6301, 117195), (6305, 117207), (6323, 118912), (6324, 118916), (6325, 118920), (6326, 118924), (6327, 118927), (6329, 118932), (6330, 118936), (6331, 118940), (6333, 118946), (6348, 118992), (6351, 119004), (6368, 119705), (6379, 119741), (6381, 119749), (6383, 119757), (6384, 119761), (6386, 119769), (6387, 119773), (6388, 119777), (6389, 119781), (6390, 119785), (6391, 119787), (6396, 119805), (6399, 119817), (6400, 119821), (6401, 119824), (6402, 119825), (6404, 119833), (6405, 119837), (6407, 119845), (6408, 119849), (6409, 119853), (6410, 119857), (6411, 119860), (6412, 119861), (6413, 119865), (6418, 119885), (6419, 119886), (6421, 119893), (6423, 119901), (6424, 119905), (6426, 119913), (6427, 119917), (6428, 119921), (6430, 119929), (6431, 119930), (6432, 119933), (6433, 119937), (6434, 119941), (6435, 119944), (6436, 119945), (6437, 119949), (6438, 119953), (6439, 119957), (6443, 119973), (6444, 119977), (6445, 119978), (6446, 119981), (6447, 119985), (6450, 119997), (6452, 120003), (6453, 120005), (6454, 120009), (6455, 120013), (6456, 120017), (6457, 120018), (6458, 120021), (6459, 120025), (6462, 120037), (6465, 120045), (6468, 120057), (6469, 120061), (6471, 120069), (6475, 120081), (6476, 120085), (6477, 120089), (6478, 120093), (6479, 120094), (6482, 120105), (6485, 120117), (6490, 120133), (6492, 120141), (6499, 120165), (6505, 120189), (6506, 120193), (6512, 120213), (6514, 120221), (6515, 120225), (6516, 120229), (6517, 120233), (6519, 120241), (6520, 120245), (6523, 120257), (6524, 120261), (6529, 120277), (6530, 120281), (6531, 120285), (6533, 120293), (6549, 121149), (6550, 121153), (6551, 121157), (6554, 121169), (6556, 121175), (6557, 121177), (6558, 121181), (6559, 121185), (6560, 121189), (6561, 121193), (6563, 121197), (6564, 121201), (6569, 121219), (6571, 121225), (6572, 121229), (6573, 121233), (6575, 121241), (6578, 121249), (6583, 121269), (6584, 121273), (6586, 121277), (6587, 121281), (6588, 121285), (6589, 121286), (6590, 121289), (6591, 121293), (6592, 121297), (6593, 121301), (6594, 122122), (6609, 122178), (6612, 122190), (6614, 122198), (6618, 122214), (6619, 122218), (6621, 122226), (6637, 122282), (6638, 122286), (6642, 122301), (6643, 122302), (6644, 122306), (6645, 122310), (6646, 122314), (6647, 122318), (6648, 122319), (6649, 122322), (6650, 122326), (6651, 122330), (6654, 122342), (6655, 122345), (6656, 122346), (6658, 122354), (6660, 122362), (6661, 122366), (6665, 123951), (6669, 123963), (6670, 123967), (6677, 123991), (6678, 123995), (6679, 123999), (6680, 124003), (6681, 124007), (6683, 124015), (6684, 124019), (6685, 124023), (6686, 124024), (6688, 124031), (6689, 124035), (6690, 124038), (6691, 124039), (6694, 124051), (6695, 124055), (6696, 124059), (6697, 124061), (6698, 124063), (6699, 124067), (6701, 124074), (6702, 124075), (6708, 124095)],
    "Anthony_Sinisuka_GINTING_Anders_ANTONSEN_Indonesia_Masters_2020_Final": [(154, 25104), (307, 28318), (313, 28342), (334, 28414), (395, 29820), (436, 29964), (463, 30933), (464, 30937), (486, 31013), (518, 31642), (519, 31645), (520, 31649), (560, 32982), (561, 32986), (562, 32990), (563, 32994), (564, 32995), (565, 32998), (587, 33078), (646, 34086), (647, 34087), (665, 34146), (732, 35346), (735, 35354), (848, 37687), (875, 38285), (905, 38834), (906, 38837), (907, 38838), (908, 38842), (969, 40251), (970, 40255), (1104, 43077), (1206, 44467), (1209, 44479), (1210, 44483), (1211, 44487), (1295, 46621), (1439, 47850), (1440, 47854), (1441, 47858), (1442, 47862), (1443, 47865), (1444, 47866), (1445, 47870), (1571, 49658), (1611, 51229), (1645, 51349), (1700, 51541), (1701, 51545), (1702, 51549), (1703, 51553), (1725, 52907), (1752, 56054), (1753, 56058), (1785, 56170), (1806, 56775), (1866, 59453), (1905, 59589), (1917, 59629), (1918, 59633), (1919, 59637), (1960, 59785), (1985, 59873), (2009, 61567), (2011, 61572), (2051, 62347), (2074, 62434), (2102, 62530), (2103, 62534), (2106, 62542), (2127, 66431), (2128, 66434), (2133, 66454), (2165, 67296), (2169, 67312), (2170, 67316), (2171, 67319), (2172, 67320), (2173, 67324), (2174, 67328), (2175, 67332), (2177, 67720), (2179, 67728), (2180, 67731), (2181, 67732), (2182, 67736), (2183, 67740), (2214, 68362), (2215, 68366), (2216, 68370), (2217, 68374), (2219, 68381), (2221, 68386), (2222, 68390), (2223, 68392), (2224, 68394), (2225, 68398), (2234, 68959), (2236, 68967), (2237, 68970), (2238, 68971), (2239, 68975), (2253, 70731), (2266, 70775), (2267, 70779), (2268, 70783), (2269, 70787), (2294, 71292), (2295, 71296), (2298, 71304), (2299, 71308), (2300, 71312), (2317, 72421), (2321, 72433), (2325, 72448), (2327, 72453), (2351, 72541), (2401, 73398), (2402, 73402), (2403, 73404), (2404, 73406), (2409, 73422), (2448, 74647), (2449, 74649), (2457, 74677), (2458, 74681), (2459, 74685), (2460, 74689), (2488, 74785), (2500, 74825), (2512, 74873), (2516, 74885), (2517, 74889), (2520, 74901), (2529, 74937), (2530, 74938), (2531, 74941), (2543, 74985), (2544, 74988), (2545, 74989), (2547, 74997), (2550, 75005), (2587, 75133), (2647, 75981), (2674, 77080), (2731, 78069), (2732, 78073), (2737, 78093), (2738, 78094), (2739, 78097), (2741, 78105), (2743, 78113), (2744, 78117), (2746, 78121), (2757, 78165), (2758, 78167), (2759, 78169), (2760, 78173), (2769, 78205), (2775, 78229), (2776, 78233), (2793, 79745), (2794, 79747), (2795, 79751), (2806, 79791), (2807, 79792), (2808, 79795), (2809, 79799), (2810, 79803), (2812, 79811), (2819, 79835), (2820, 79839), (2822, 79847), (2884, 81081), (2885, 81085), (2886, 81089), (2887, 81091), (2890, 81101), (2891, 81102), (2892, 81105), (2899, 81129), (2900, 81133), (2914, 81185), (2915, 81186), (2916, 81189), (2938, 81963), (3036, 85171), (3041, 85191), (3044, 85201), (3061, 85255), (3078, 85315), (3079, 85319), (3082, 85330), (3083, 85331), (3085, 85339), (3086, 85343), (3087, 85347), (3088, 85348), (3089, 85351), (3096, 85379), (3106, 85411), (3117, 85455), (3144, 85551), (3147, 85563), (3156, 85595), (3158, 85603), (3209, 87768), (3224, 87820), (3274, 88000), (3307, 88851), (3308, 88854), (3309, 88855), (3310, 88859), (3311, 88863), (3314, 88875), (3315, 88879), (3316, 88880), (3317, 88883), (3331, 88937), (3332, 88939), (3346, 89496), (3347, 89499), (3360, 89543), (3362, 89551), (3363, 89552), (3370, 89575), (3397, 89671), (3398, 89675), (3399, 89679), (3400, 89683), (3409, 90872), (3410, 90876), (3411, 90880), (3455, 91532), (3456, 91536), (3467, 91576), (3468, 91580), (3470, 91588), (3471, 91589), (3473, 91596), (3475, 91604), (3508, 92375), (3509, 92378), (3510, 92379), (3511, 92383), (3512, 92387), (3534, 92459), (3561, 92547), (3587, 94367), (3588, 94369), (3589, 94371), (3590, 94375), (3665, 95954), (3667, 95962), (3668, 95966), (3671, 95974), (3672, 95978), (3673, 95982), (3676, 95994), (3677, 95998), (3678, 96002), (3687, 96034), (3688, 96035), (3714, 96126), (3736, 96202), (3755, 97963), (3756, 97964), (3757, 97967), (3758, 97971), (3790, 98654), (3837, 99398), (3838, 99400), (3841, 99412), (3848, 99436), (3864, 100795), (3865, 100799), (3867, 100807), (3880, 100855), (3893, 100903), (3903, 100939), (3904, 100943), (3905, 100947), (3912, 101951), (3919, 101975), (3920, 101979), (3923, 101988), (3924, 101991), (3925, 101995), (3944, 102063), (3945, 102067), (3946, 102071), (3953, 102094), (3954, 102095), (3955, 102099), (3961, 102123), (3986, 102213), (3988, 102219), (3989, 102222), (3997, 102247), (4012, 102303), (4047, 106397), (4106, 106601), (4154, 107306), (4155, 107310), (4156, 107312), (4178, 107948), (4215, 108080), (4216, 108082), (4217, 108084), (4218, 108088), (4219, 108092), (4220, 108096), (4253, 108209), (4254, 108212), (4258, 108228), (4260, 108232), (4261, 108236), (4263, 108244), (4304, 110368), (4330, 110868), (4353, 110953), (4354, 110956), (4355, 110960), (4365, 110996), (4479, 113007), (4544, 115006), (4566, 116118), (4616, 117076), (4617, 117080), (4618, 117084), (4619, 117088), (4620, 117091), (4677, 117834), (4678, 117835), (4707, 120697), (4709, 120705), (4711, 120713), (4713, 120721), (4755, 120881), (4758, 120893), (4759, 120897), (4767, 120925), (4771, 120940), (4773, 120945), (4774, 120949), (4779, 120965), (4782, 120977), (4783, 120979), (4784, 120981), (4785, 120985), (4794, 121021), (4809, 121077), (4934, 122940), (4941, 122964), (4951, 122996), (4952, 123000), (4953, 123004), (4963, 124237), (4964, 124241), (4965, 124245), (4966, 124249), (4967, 124252), (4968, 124253), (4969, 124257), (5031, 125754), (5092, 126443), (5096, 126455), (5097, 126459), (5106, 126870), (5107, 126874), (5108, 126878), (5109, 126882), (5110, 126885), (5111, 126886), (5112, 126890), (5113, 126894), (5114, 126898), (5123, 127546), (5124, 127550), (5128, 127562), (5129, 127566), (5130, 127570), (5131, 127571), (5132, 127574), (5133, 127578), (5134, 127582), (5137, 127594), (5138, 127595), (5139, 127598), (5140, 127602), (5153, 127650), (5154, 127652), (5167, 127700), (5168, 127702), (5169, 127706), (5170, 127710), (5173, 127718), (5174, 127722), (5175, 127726), (5239, 129307), (5282, 129856), (5283, 129857), (5284, 129860), (5285, 129864), (5286, 129868), (5297, 129908), (5298, 129912), (5334, 130774), (5335, 130778), (5336, 130782), (5337, 130785), (5338, 130786), (5339, 130790), (5340, 130794), (5341, 130798), (5342, 130801), (5343, 130802), (5344, 130806), (5345, 130810), (5346, 130814)],
    "Anthony_Sinisuka_GINTING_Viktor_AXELSEN _Indonesia_Masters_2020_SemiFinals": [(101, 14010), (111, 14044), (149, 14176), (175, 14687), (177, 14695), (181, 14707), (212, 15122), (218, 15146), (341, 16097), (379, 17128), (390, 17168), (391, 17172), (394, 17180), (435, 17328), (586, 18866), (622, 18994), (645, 20012), (646, 20016), (647, 20020), (670, 20100), (671, 20104), (672, 20105), (677, 20120), (706, 21183), (714, 21211), (715, 21212), (717, 21219), (730, 21267), (739, 21295), (752, 21339), (768, 21399), (804, 21519), (841, 22763), (894, 24252), (934, 24392), (954, 24468), (955, 24472), (961, 24492), (988, 25182), (989, 25183), (995, 25203), (999, 25215), (1002, 25224), (1003, 25227), (1007, 25243), (1008, 25244), (1013, 26405), (1014, 26409), (1015, 26413), (1016, 26417), (1042, 26513), (1043, 26517), (1136, 27427), (1138, 27433), (1143, 30222), (1164, 31943), (1182, 32007), (1230, 32837), (1249, 34460), (1250, 34464), (1270, 34532), (1273, 34540), (1274, 34544), (1275, 34546), (1276, 34548), (1277, 34552), (1281, 34568), (1282, 34572), (1284, 34576), (1285, 34580), (1296, 34620), (1299, 34632), (1300, 34636), (1301, 34637), (1302, 34640), (1303, 34644), (1304, 34648), (1374, 35898), (1375, 35902), (1378, 35912), (1379, 35914), (1380, 35918), (1381, 35922), (1386, 35938), (1387, 35942), (1389, 35950), (1390, 35954), (1405, 36006), (1406, 36008), (1407, 36010), (1408, 36014), (1409, 36018), (1410, 36022), (1411, 37035), (1412, 37039), (1458, 37794), (1477, 37862), (1478, 37866), (1480, 37874), (1481, 37878), (1482, 37882), (1483, 37884), (1484, 37886), (1510, 38968), (1607, 40260), (1668, 40468), (1671, 40476), (1685, 40532), (1686, 40536), (1717, 40636), (1735, 40699), (1737, 40704), (1789, 41998), (1814, 43060), (1839, 43822), (1840, 43826), (1861, 43894), (1917, 45061), (1939, 45137), (1982, 46685), (1983, 46689), (1984, 46693), (2010, 47902), (2014, 47914), (2029, 48529), (2040, 48569), (2041, 48573), (2042, 48577), (2043, 48581), (2046, 52121), (2049, 52133), (2064, 52181), (2077, 52480), (2079, 52488), (2080, 52492), (2084, 52504), (2085, 52508), (2087, 52516), (2088, 52520), (2089, 52524), (2090, 52527), (2091, 52528), (2092, 52532), (2093, 52536), (2096, 52544), (2111, 52600), (2112, 52604), (2114, 52612), (2115, 52616), (2116, 52620), (2117, 52624), (2118, 52628), (2120, 52636), (2121, 52639), (2122, 52640), (2123, 52644), (2124, 52648), (2126, 52656), (2137, 52692), (2139, 52700), (2141, 52704), (2157, 52760), (2158, 52764), (2159, 52768), (2160, 53286), (2176, 53341), (2181, 53623), (2182, 53627), (2183, 53631), (2185, 53638), (2189, 53651), (2190, 53655), (2192, 53663), (2194, 53671), (2196, 53679), (2197, 53682), (2198, 53683), (2199, 53687), (2200, 53691), (2201, 53695), (2202, 53699), (2204, 53706), (2205, 53707), (2206, 53711), (2207, 53715), (2208, 53719), (2211, 53730), (2212, 53731), (2213, 53735), (2214, 53739), (2215, 53743), (2216, 53747), (2217, 53751), (2218, 53755), (2219, 53759), (2228, 53791), (2246, 53851), (2247, 53855), (2248, 53859), (2249, 53863), (2263, 53911), (2264, 53915), (2268, 54736), (2271, 54744), (2272, 54748), (2273, 54752), (2275, 54760), (2281, 54780), (2294, 54824), (2295, 54828), (2297, 54836), (2303, 54856), (2305, 54864), (2306, 54868), (2307, 54872), (2308, 54876), (2309, 54879), (2310, 54880), (2311, 54884), (2314, 54896), (2316, 54904), (2317, 54907), (2318, 54908), (2319, 54912), (2321, 54916), (2322, 54920), (2332, 54952), (2356, 55036), (2357, 55040), (2359, 55048), (2360, 55050), (2363, 55060), (2364, 55064), (2385, 55573), (2386, 55577), (2387, 55581), (2395, 55609), (2400, 55629), (2401, 55633), (2402, 55637), (2403, 55641), (2404, 55645), (2405, 55646), (2406, 55649), (2408, 55657), (2409, 55661), (2430, 56447), (2432, 56453), (2434, 56461), (2460, 56924), (2464, 56936), (2476, 56980), (2488, 57024), (2490, 57032), (2491, 57036), (2502, 57072), (2504, 57080), (2505, 57084), (2508, 57092), (2511, 57104), (2519, 57132), (2544, 58302), (2545, 58306), (2546, 58310), (2547, 58314), (2548, 58318), (2550, 58324), (2551, 58326), (2552, 58327), (2554, 58334), (2555, 58338), (2556, 58342), (2557, 58346), (2562, 58362), (2563, 58366), (2564, 58367), (2565, 58370), (2566, 58374), (2567, 58378), (2573, 58398), (2575, 58406), (2576, 59513), (2589, 59557), (2591, 59565), (2592, 59568), (2593, 59569), (2594, 59573), (2598, 59589), (2599, 59593), (2600, 59597), (2604, 59613), (2605, 59617), (2606, 59621), (2607, 59625), (2608, 59629), (2609, 59633), (2610, 59637), (2611, 59640), (2612, 59641), (2614, 59649), (2615, 59653), (2619, 59666), (2620, 59669), (2623, 59681), (2627, 59697), (2644, 60382), (2646, 60387), (2647, 60390), (2664, 60849), (2684, 60921), (2688, 60937), (2693, 60953), (2694, 60957), (2698, 62272), (2699, 62275), (2700, 62276), (2701, 62280), (2706, 62297), (2707, 62300), (2709, 62308), (2730, 62672), (2731, 62676), (2732, 62680), (2736, 62692), (2760, 63476), (2763, 63488), (2764, 63489), (2767, 63500), (2770, 63512), (2771, 63516), (2772, 63520), (2773, 63524), (2774, 63527), (2775, 63528), (2776, 63532), (2777, 63536), (2778, 63540), (2783, 63556), (2785, 63564), (2810, 63955), (2812, 63962), (2813, 63963), (2814, 63967), (2848, 64083), (2855, 64106), (2871, 64163), (2874, 64175), (2875, 64179), (2885, 64215), (2890, 64231), (2891, 64235), (2894, 64243), (2896, 64251), (2908, 64299), (2909, 64303), (2910, 64307), (2929, 64372), (2932, 64383), (2934, 64391), (2935, 64395), (2939, 64407), (2940, 64411), (2941, 64415), (2942, 64419), (2943, 64423), (2944, 64427), (2945, 64431), (2946, 64435), (2947, 64439), (2952, 64459), (2953, 64463), (2956, 64475), (2957, 64479), (2958, 64483), (2977, 65950), (2979, 65956), (2981, 65964), (3001, 68534), (3002, 68538), (3003, 68542), (3007, 68554), (3008, 68556), (3013, 68570), (3014, 68574), (3025, 68614), (3027, 68622), (3040, 68670), (3041, 68674), (3044, 68684), (3045, 68686), (3081, 72324), (3082, 72328), (3092, 72360), (3093, 72362), (3094, 72364), (3104, 72404), (3105, 72408), (3106, 72720), (3107, 72724), (3117, 72759), (3118, 72760), (3119, 72764), (3120, 72768), (3132, 72812), (3133, 72816), (3134, 72820), (3135, 72824), (3136, 72825), (3137, 72828), (3138, 72832), (3140, 72840), (3141, 72844), (3146, 72860), (3147, 72864), (3166, 75879), (3171, 75895), (3172, 75899), (3173, 75903), (3174, 76210), (3175, 76214), (3202, 76306), (3239, 77934), (3240, 77938), (3241, 77942), (3254, 77986), (3258, 78002), (3260, 78008), (3263, 78018), (3264, 78022), (3265, 78026), (3266, 78030), (3267, 78034), (3268, 78038), (3269, 78042), (3270, 78046), (3271, 78047), (3273, 78054), (3274, 78058), (3282, 78086), (3283, 78090), (3297, 78142), (3298, 78146), (3299, 78150), (3300, 78154), (3301, 78158), (3302, 78160), (3303, 78162), (3305, 78170), (3306, 78174)],
    "CHEN_Long_CHOU_Tien_Chen_World_Tour_Finals_Group_Stage": [(152, 12688), (354, 14369), (355, 14373), (356, 14377), (357, 14381), (373, 14838), (374, 14842), (375, 14846), (376, 14850), (398, 14922), (542, 16190), (741, 18381), (745, 18944), (755, 18976), (756, 18980), (882, 20633), (1007, 21069), (1012, 21089), (1086, 21986), (1120, 23040), (1263, 25046), (1275, 25094), (1379, 26027), (1380, 26030), (1408, 26825), (1409, 26829), (1410, 26833), (1411, 26837), (1440, 26936), (1441, 26937), (1442, 26941), (1555, 27349), (1583, 27453), (1629, 27629), (1630, 27630), (1678, 27797), (1679, 27801), (1681, 29254), (1682, 29258), (1683, 29262), (1684, 29266), (1743, 31482), (1841, 33396), (1856, 33449), (1857, 33452), (1858, 33456), (1903, 34528), (1904, 34532), (1929, 35047), (1930, 35051), (1931, 35055), (2098, 36989), (2099, 36993), (2100, 36997), (2139, 37129), (2303, 38982), (2357, 40589), (2382, 41082), (2383, 41086), (2384, 41090), (2496, 41486), (2585, 41810), (2586, 41814), (2604, 41874), (2608, 42503), (2609, 42507), (2610, 42511), (2611, 42515), (2925, 50018), (2991, 50258), (3063, 51193), (3064, 51197), (3065, 51201), (3066, 51205), (3075, 51596), (3076, 51600), (3090, 51646), (3119, 51743), (3126, 51768), (3143, 51828), (3144, 51829), (3494, 55269), (3495, 55270), (3496, 55274), (3497, 55278), (3498, 55282), (3545, 57286), (3550, 57302), (3575, 57394), (3599, 57482), (3601, 57486), (3610, 57975), (3644, 58095), (3675, 58211), (3734, 58812), (3735, 58816), (3882, 61140), (3883, 61142), (3884, 61146), (3996, 62613), (4100, 65079), (4227, 67921), (4228, 67925), (4229, 67929), (4230, 67930), (4231, 67933), (4234, 67945), (4333, 70562), (4334, 70566), (4335, 70567), (4363, 71014), (4486, 72343), (4521, 72467), (4944, 76719), (4951, 76743), (4952, 76747), (4953, 76751), (4954, 76755), (4955, 76759), (4956, 76761), (4957, 76763), (5009, 78539), (5010, 78543), (5011, 78547), (5012, 78550), (5013, 78551), (5014, 78555), (5037, 78639), (5038, 78643), (5044, 78667), (5045, 78671), (5046, 78673), (5047, 78675), (5048, 78679), (5049, 78683), (5050, 78687), (5051, 78691), (5052, 78695), (5053, 78699), (5116, 80679), (5251, 83003), (5252, 83005), (5307, 83197), (5391, 83493), (5421, 83597), (5508, 85854), (5509, 85858), (5575, 87533), (5576, 87535), (5577, 87537), (5681, 88747)],
    "CHOU_Tien_Chen_Anders_ANTONSEN_Fuzhou_Open_2019_Semi-finals": [(107, 10672), (290, 11858), (316, 11954), (321, 11970), (354, 12086), (355, 12090), (356, 12094), (357, 12098), (1098, 21814), (2254, 41515), (2255, 41519), (2256, 41521), (2257, 41523), (3014, 51306), (3447, 56738), (3448, 56742), (3449, 56746), (3546, 60984), (3547, 60988), (3659, 61770), (4616, 75989), (4617, 75993), (4618, 75997), (4701, 77082), (4702, 77086)],
    "Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_Finals": [(290, 13908), (366, 14176), (502, 16310), (753, 18151), (754, 18155), (755, 18159), (756, 18160), (861, 18527), (943, 18819), (944, 18823), (978, 18947), (1178, 20164), (1377, 21387), (1448, 21647), (2097, 25969), (2098, 25971), (2574, 33147), (2583, 33176), (2584, 33179), (2790, 35121), (2791, 35125), (2962, 36751), (2986, 36838), (2987, 36839), (3025, 36979), (3026, 36980), (3059, 37099), (3155, 37459), (3384, 40606), (3556, 41723), (3558, 41731), (3821, 44524), (3925, 48598), (3926, 48602), (4146, 51498), (4147, 51502), (4148, 51506), (4465, 55133), (4467, 55137), (4486, 55201), (4529, 55361), (4541, 55401), (4702, 56863), (4703, 56867), (4705, 56875), (4755, 57054), (4798, 58015), (4836, 58833), (4938, 59568), (5024, 60637), (5025, 60641), (5026, 60645), (5070, 60797), (5217, 63926), (5218, 63927), (5392, 65467), (5393, 65471), (5557, 67682), (5672, 68092), (5739, 68328), (5740, 68330), (5772, 68444), (5873, 70294), (5934, 70510), (6063, 71532), (6456, 75054), (6607, 75582), (6608, 75586), (6609, 75587), (6611, 75594), (6612, 75598), (6613, 75602), (6678, 77821), (6692, 77869), (6693, 77873), (6736, 78347), (6827, 78675), (6835, 78703), (7136, 84050), (7161, 84138), (7309, 84666), (7310, 84670), (7311, 84674), (7473, 87416), (7795, 93788), (7796, 93789), (7797, 93792), (7870, 94984), (8181, 97395), (8526, 101817), (8581, 102016), (8582, 102017), (9139, 109193), (9518, 113961), (9780, 115454), (9792, 115498), (9793, 115502), (9794, 115506), (9881, 116948), (9911, 117056), (9987, 117327), (9988, 117328), (9989, 117332), (9990, 117336), (9991, 117340), (10019, 118981), (10052, 119097), (10175, 119529), (10176, 119531), (10177, 119533), (10293, 119941), (10481, 123147), (10482, 123151), (10818, 126614), (10907, 127938), (10962, 128788), (11167, 131684), (11233, 133342)],
}

total = sum(len(v) for v in BAD_FRAMES.values())
print(f"Bad frames to re-extract: {total} across {len(BAD_FRAMES)} matches")
for match, frames in BAD_FRAMES.items():
    print(f"  {match[:60]:60s} — {len(frames)} frames")

## 3. Load Models (GDINO + YOLO)

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image as _PIL
from ultralytics import YOLO

_DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_MODEL_ID = 'IDEA-Research/grounding-dino-tiny'

print(f"Loading GDINO-tiny on {_DEVICE}...")
_gdino_proc  = AutoProcessor.from_pretrained(_MODEL_ID)
_gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(_MODEL_ID).to(_DEVICE)
_gdino_model.eval()
print("  GDINO ready")

_yolo = YOLO('yolov8s-pose.pt')
print("  YOLOv8s-pose ready")

## 4. Extraction Helpers (tighter boundary)

In [ ]:
def _iou(b1, b2):
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter + 1e-9
    return inter / union


def _in_court_region(box, W, H):
    cx = (box[0] + box[2]) / 2
    cy = (box[1] + box[3]) / 2
    box_h = box[3] - box[1]
    return (W * _X_MIN_FRAC <= cx <= W * _X_MAX_FRAC and
            H * _Y_MIN_FRAC <= cy <= H * _Y_MAX_FRAC and
            box_h >= H * _MIN_H_FRAC)


@torch.no_grad()
def _gdino_detect(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H, W = bgr.shape[:2]
    inp = _gdino_proc(
        images=_PIL.fromarray(rgb), text=_GDINO_PROMPT, return_tensors='pt'
    ).to(_DEVICE)
    out = _gdino_model(**inp)
    res = _gdino_proc.post_process_grounded_object_detection(
        outputs=out, input_ids=inp['input_ids'],
        threshold=_BOX_THRESH, text_threshold=_TXT_THRESH,
        target_sizes=[(H, W)]
    )[0]
    return res['boxes'].cpu().numpy(), res['scores'].cpu().numpy()


def _extract_frame(bgr):
    """Extract (2, 34) skeleton with the tighter court boundary."""
    H, W = bgr.shape[:2]
    g_boxes_all, _ = _gdino_detect(bgr)

    g_boxes = np.array([b for b in g_boxes_all if _in_court_region(b, W, H)]) \
              if len(g_boxes_all) else np.zeros((0, 4))

    res = _yolo(bgr, verbose=False)[0]
    out = np.zeros((2, 34), dtype=np.float32)
    if res is None or res.keypoints is None or res.boxes is None:
        return out

    y_boxes = res.boxes.xyxy.cpu().numpy()
    y_cls   = res.boxes.cls.cpu().numpy()
    y_kpts  = res.keypoints.xy.cpu().numpy()
    y_confs = res.boxes.conf.cpu().numpy()

    kept_kpts, kept_confs, kept_cy = [], [], []
    for i, (yb, yc, yconf) in enumerate(zip(y_boxes, y_cls, y_confs)):
        if int(yc) != 0:
            continue
        if not _in_court_region(yb, W, H):
            continue
        max_iou = max((_iou(yb, gb) for gb in g_boxes), default=0.0) if len(g_boxes) else 0.0
        if max_iou >= _IOU_THRESH:
            kept_kpts.append(y_kpts[i])
            kept_confs.append(float(yconf))
            kept_cy.append(float(y_kpts[i][:, 1].mean()))

    if len(kept_kpts) < 2:
        valid_idx = [
            i for i, (yb, yc) in enumerate(zip(y_boxes, y_cls))
            if int(yc) == 0 and _in_court_region(yb, W, H)
        ]
        if len(valid_idx) >= 2:
            valid_idx.sort(key=lambda i: -y_confs[i])
            top2 = valid_idx[:2]
            kept_kpts  = [y_kpts[j]  for j in top2]
            kept_cy    = [float(y_kpts[j][:, 1].mean()) for j in top2]
            kept_confs = [float(y_confs[j]) for j in top2]
        else:
            return out

    if len(kept_kpts) > 2:
        order = np.argsort(kept_confs)[::-1][:2]
        kept_kpts = [kept_kpts[j] for j in order]
        kept_cy   = [kept_cy[j]   for j in order]

    ys = np.argsort(kept_cy)
    p0, p1 = kept_kpts[ys[0]], kept_kpts[ys[1]]
    out[0, :17] = p0[:, 0];  out[1, :17] = p0[:, 1]
    out[0, 17:] = p1[:, 0];  out[1, 17:] = p1[:, 1]
    return out

print("Extraction helpers ready.")

## 5. Re-extract Bad Frames

Loads each match's `skeletons.npy`, patches only the bad frame slices, and saves back.

In [ ]:
total = sum(len(v) for v in BAD_FRAMES.values())
print(f"Re-extracting {total} hardcoded bad frames across {len(BAD_FRAMES)} matches")
print(f"Court boundary: x=[{X_MIN},{X_MAX}], y=[{Y_MIN},{Y_MAX}]\n")

for match_name, bad_list in BAD_FRAMES.items():
    sk_path = SS_SKELETONS_GDINO / match_name / 'skeletons.npy'
    if not sk_path.exists():
        print(f"[SKIP] {match_name}: skeletons.npy not found")
        continue

    sk = np.load(sk_path)  # (2, T, 34)
    frame_dir = SS_FRAMES / match_name
    if not frame_dir.exists():
        print(f"[SKIP] {match_name}: no frames dir")
        continue

    print(f"{match_name[:60]}: {len(bad_list)} frames...")
    fixed = zeroed = missing = 0

    for t_idx, frame_num in tqdm(bad_list, desc='  Frames'):
        fp = frame_dir / f'frame_{frame_num:06d}.jpg'
        if not fp.exists():
            fp = frame_dir / f'frame_{frame_num:06d}.png'
        if not fp.exists():
            missing += 1
            continue

        bgr = cv2.imread(str(fp))
        if bgr is None:
            missing += 1
            continue

        new_skel = _extract_frame(bgr)
        sk[:, t_idx, :] = new_skel

        if new_skel.sum() == 0:
            zeroed += 1
        else:
            fixed += 1

    np.save(sk_path, sk)
    print(f"  Done: {fixed} fixed, {zeroed} zeroed, {missing} missing frames\n")

print("=" * 60)
print("All done.")

## 6. Save Updated Skeletons to Drive (Colab only)

In [ ]:
if IN_COLAB:
    import shutil

    DRIVE_DIR = Path('/content/drive/MyDrive/FineBadminton')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    out_zip = DRIVE_DIR / 'shuttleset_skeletons_gdino.zip'

    n_files = len(list(SS_SKELETONS_GDINO.rglob('*.npy'))) if SS_SKELETONS_GDINO.exists() else 0
    print(f"Skeletons to archive: {n_files} .npy files")

    if n_files == 0:
        print("[WARN] No skeletons found.")
    else:
        print(f"Zipping to {out_zip} ...")
        shutil.make_archive(str(out_zip).replace('.zip', ''), 'zip',
                            SS_SKELETONS_GDINO.parent, SS_SKELETONS_GDINO.name)
        size_mb = out_zip.stat().st_size / 1e6
        print(f"Saved: {out_zip} ({size_mb:.1f} MB, {n_files} skeletons)")
else:
    print("Local run — skeletons already on disk, no Drive save needed")